# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [6]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.appName("PatentJoinRDD").getOrCreate()
sc = spark.sparkContext


# Loading datasets as DataFrames

In [7]:
patents = spark.read.csv("apat63_99.txt.gz", header=True)
citations = spark.read.csv("cite75_99.txt.gz", header=True)


# Converting to RDDs

In [8]:
patents_rdd = patents.rdd.map(lambda row: (row.PATENT, row.asDict()))

In [9]:
citations_rdd = citations.rdd.map(lambda row: (row.CITING, row.CITED))

# Preparing lookup: (PATENT, STATE)

In [10]:
patent_states = patents.rdd.map(lambda row: (row.PATENT, row.POSTATE))


# Join citations with cited patent to get CITED_STATE


In [11]:
cited_join = citations_rdd.map(lambda x: (x[1], x[0])) \
                          .join(patent_states) \
                          .map(lambda x: (x[1][0], (x[0], x[1][1])))


# Joining to get CITING_STATE


In [12]:
citing_join = cited_join.join(patent_states) \
                        .map(lambda x: (x[0], x[1][0][0], x[1][0][1], x[1][1]))


# Flaging self-state citations


In [13]:
self_flags = citing_join.map(lambda x: (x[0], 1 if x[2] == x[3] and x[2] is not None else 0))


# Aggregating self-state counts


In [14]:
self_counts = self_flags.reduceByKey(lambda a, b: a + b)


# Joining counts back with full patents data


In [15]:
patents_with_self = patents_rdd.leftOuterJoin(self_counts) \
                               .map(lambda x: {**x[1][0], "self_state_count": x[1][1] if x[1][1] is not None else 0})


# Obtaining Top 10 patents by self-state citations


In [16]:
top10_rdd = patents_with_self.sortBy(lambda x: (-int(x["self_state_count"]), int(x["PATENT"])))
top10 = top10_rdd.take(10)

# Displaying in the notebook


In [17]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

top10_pd = pd.DataFrame(top10)
top10_pd

,PATENT,GYEAR,GDATE,APPYEAR,COUNTRY,POSTATE,ASSIGNEE,ASSCODE,CLAIMS,NCLASS,CAT,SUBCAT,CMADE,CRECEIVE,RATIOCIT,GENERAL,ORIGINAL,FWDAPLAG,BCKGTLAG,SELFCTUB,SELFCTLB,SECDUPBD,SECDLWBD,self_state_count
0,5959466,1999,14515,1997,US,CA,5310,2,None,326,4,46,159,0,1,None,0.6186,None,4.8868,0.0455,0.044,None,None,125
1,5983822,1999,14564,1998,US,TX,569900,2,None,114,5,55,200,0,0.995,None,0.7201,None,12.45,0,0,None,None,103
2,6008204,1999,14606,1998,US,CA,749584,2,None,514,3,31,121,0,1,None,0.7415,None,5,0.0085,0.0083,None,None,100
3,5952345,1999,14501,1997,US,CA,749584,2,None,514,3,31,118,0,1,None,0.7442,None,5.1102,0,0,None,None,98
4,5958954,1999,14515,1997,US,CA,749584,2,None,514,3,31,116,0,1,None,0.7397,None,5.181,0,0,None,None,96
5,5998655,1999,14585,1998,US,CA,None,1,None,560,1,14,114,0,1,None,0.7387,None,5.1667,None,None,None,None,96
6,5936426,1999,14466,1997,US,CA,5310,2,None,326,4,46,178,0,1,None,0.58,None,11.2303,0.0765,0.073,None,None,94
7,5739256,1998,13983,1995,US,CA,70060,2,15,528,1,15,453,0,1,None,0.8232,None,15.1104,0.1124,0.1082,None,None,90
8,5913855,1999,14417,1997,US,CA,733846,2,None,606,3,32,242,0,1,None,0.7403,None,8.3595,0,0,None,None,90
9,5925042,1999,14445,1997,US,CA,733846,2,None,606,3,32,242,0,1,None,0.7382,None,8.3471,0,0,None,None,90
